# Նեյրոնային ցանց՝ զրոյից. լուծի՛ր XOR-ը

*1991 Academy · Լաբորատորիա*

XOR-ը գծայնորեն բաժանելի չէ — ոչ մի առանձին նեյրոն չի կարող այն լուծել (այդ հայտնագործությունը սառեցրեց նեյրոնային-ցանցերի հետազոտությունը մեկ տասնամյակով)։ Մեկ թաքնված շերտով ցանցը կարող է։ Դու պատրաստվում ես կառուցել այն. forward անցում, հետադարձ տարածում, թարմացումներ։ Ոչ մի գրադարան։

**Ճարտարապետություն.** 2 մուտք → 4 թաքնված (tanh) → 1 ելք (sigmoid), քառակուսի-սխալ կորուստ, ամեն-նմուշ SGD։

**Forward**, (x1, x2) մուտքի համար.

```
h[j] = tanh(W1[j][0]·x1 + W1[j][1]·x2 + b1[j])     j = 0..3
z    = Σ W2[j]·h[j] + b2
p    = 1 / (1 + e^(−z))          # the prediction
```

**Backward** (շղթայի կանոն, ճիշտ ինչպես backprop դասում).

```
dz     = 2·(p − y) · p·(1 − p)
dW2[j] = dz · h[j]        db2 = dz
dh[j]  = dz · W2[j]
dpre[j]= dh[j] · (1 − h[j]²)      # tanh derivative
dW1[j][0] = dpre[j]·x1    dW1[j][1] = dpre[j]·x2    db1[j] = dpre[j]
```

Թարմացրո՛ւ ամեն պարամետր. `param −= lr · grad` ամեն նմուշից հետո։ Գրանցի՛ր 4 XOR նմուշների միջին կորուստը դարաշրջանում մեկ անգամ։ Վերադարձրո՛ւ `{"predict": predict, "loss_history": loss_history}`, որտեղ `predict(x1, x2)`-ը գործարկում է քո forward անցումը։

Նոթբուքը նկարում է քո ցանցի ելքը ամբողջ մուտքային քառակուսու վրայով, գումարած քո կորստի կորը։ Չորս անկյունների բաժանվելը դիտելը թաքնված շերտի *տարածություն ծալելը* դիտելն է։

> 💾 **Աշխատանքդ պահվում է.** Colab-ում վերբեռնված (**File → Upload notebook**) այս նոթբուքը գտնվում է քո Google Drive-ում՝ *Colab Notebooks* թղթապանակում։ Կարող ես փակել այն և շարունակել ավելի ուշ։
>
> ✅ Երբ բոլոր թեստերն անցնեն, վերադարձի՛ր 1991 Academy և սեղմի՛ր **Ես լուծեցի ✓**։

## Քո կոդը

Գրի՛ր լուծումդ ներքևի բջջում և գործարկի՛ր այն. սեղմի՛ր ▶ կամ Shift+Enter։

In [ ]:
# Build and train a 2-4-1 network on XOR.
# Return {"predict": predict, "loss_history": loss_history}.
def train_xor():
    X = [[0, 0], [0, 1], [1, 0], [1, 1]]
    Y = [0, 1, 1, 0]
    # fixed starting weights: everyone converges the same way
    W1 = [[0.5, -0.4], [0.3, 0.8], [-0.6, 0.2], [0.7, -0.3]]
    b1 = [0.1, -0.2, 0.05, 0.15]
    W2 = [0.4, -0.5, 0.6, 0.3]
    b2 = 0.05
    lr, epochs = 0.5, 4000
    loss_history = []

    def forward(x1, x2):
        # return (h, p): 4 hidden tanh activations and the sigmoid output
        pass

    for e in range(epochs):
        loss = 0.0
        for s in range(4):
            # forward, accumulate (p - y)**2 into loss,
            # backward (formulas in the brief), update all params
            pass
        loss_history.append(loss / 4)

    def predict(x1, x2):
        return forward(x1, x2)[1]

    return {"predict": predict, "loss_history": loss_history}

## Ստուգի՛ր լուծումդ

Գործարկի՛ր ներքևի բջիջը։ Եթե թեստը չանցնի, ուղղի՛ր կոդդ, նորից գործարկի՛ր կոդի բջիջը, հետո՝ այս մեկը։

In [ ]:
#@title ▶ Գործարկել թեստերը { display-mode: "form" }
import json, math, traceback
MSG = {
 "all_pass": "🎉 Բոլոր {n} թեստերն անցան։ Վերադարձի՛ր 1991 Academy և սեղմի՛ր «Ես լուծեցի ✓»։",
 "some_pass": "{p}/{n} թեստ է անցնում։ Շարունակի՛ր։",
 "none_ran": "Ոչ մի թեստ չաշխատեց։ Ֆունկցիայիդ անունը {fn} է՞։",
 "expected": "սպասվում էր {e}, ստացվեց {a}",
 "error": "💥 Թեստերը կանգ առան սխալով՝",
 "not_run": "Գործարկե՞լ ես վերևի կոդի բջիջը։ Թեստերը տեսնում են միայն գործարկված կոդը։"
}

def _report(results, error, fn):
    for name, ok, expected, actual in results:
        print(("✓ " if ok else "✗ ") + name + ("" if ok else "   " + MSG["expected"].format(e=expected, a=actual)))
    if error:
        print("\n" + MSG["error"])
        print(error.rstrip())
    passed = sum(1 for r in results if r[1])
    print()
    if results and passed == len(results) and not error:
        print(MSG["all_pass"].format(n=passed))
    elif results:
        print(MSG["some_pass"].format(p=passed, n=len(results)))
    elif not error:
        print(MSG["none_ran"].format(fn=fn))

def _academy_run_tests():
    results = []

    def __check(name, actual, expected):
        try:
            ok = bool(actual == expected)
        except Exception:
            ok = False
        results.append((str(name), ok, repr(expected), repr(actual)))

    def tests():
        net = None
        for attempt in range(3):
            n = train_xor()
            if n and callable(n.get("predict")):
                ok = (n["predict"](0, 0) < 0.3 and n["predict"](1, 1) < 0.3 and
                      n["predict"](0, 1) > 0.7 and n["predict"](1, 0) > 0.7)
                if ok or attempt == 2:
                    net = n
                    break
        __check("returns predict + loss_history", bool(net and callable(net.get("predict")) and isinstance(net.get("loss_history"), list)), True)
        __check("(0,0) -> ~0", net["predict"](0, 0) < 0.3, True)
        __check("(1,1) -> ~0", net["predict"](1, 1) < 0.3, True)
        __check("(0,1) -> ~1", net["predict"](0, 1) > 0.7, True)
        __check("(1,0) -> ~1", net["predict"](1, 0) > 0.7, True)
        __check("loss went DOWN during training", net["loss_history"][0] > net["loss_history"][-1] * 3, True)

    error = None
    try:
        tests()
    except Exception as exc:
        error = traceback.format_exc(limit=-3)
        if isinstance(exc, NameError):
            error += "\n" + MSG["not_run"]
    _report(results, error, 'train_xor')

_academy_run_tests()

## Հուշումներ

<details><summary>Հուշում 1</summary>

Նախ գրի՛ր forward()-ը և թեստավորի՛ր մտքումդ. տրված մեկնարկային կշիռներով predict(0,0)-ն ուսուցումից առաջ պետք է վերադարձնի 0.5-ին մոտ ինչ-որ թիվ։

</details>

<details><summary>Հուշում 2</summary>

Backprop-ի կարգը կարևոր է. հաշվի՛ր dz-ն նախ, հետո dW2/db2, հետո dh → dpre → dW1/db1 — և ՄԻԱՅՆ ՀԵՏՈ կիրառի՛ր այս նմուշի բոլոր թարմացումները։

</details>

## Տե՛ս քո կոդը գործողության մեջ

Երբ թեստերն անցնեն, գործարկի՛ր այս բջիջը՝ տեսնելու, թե ինչ է անում կոդդ։

In [ ]:
#@title 📊 Պատկերել կոդդ { display-mode: "form" }
import json
import matplotlib.pyplot as plt
VIZ = json.loads(r"""{}""")
net = train_xor()
cells, lo, hi = 56, -0.25, 1.25
heat = [[net["predict"](lo + (c + 0.5) / cells * (hi - lo), hi - (r + 0.5) / cells * (hi - lo)) for c in range(cells)]
        for r in range(cells)]
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.2))
left.imshow(heat, extent=(lo, hi, lo, hi), cmap="coolwarm", vmin=0, vmax=1)
left.scatter([0, 1, 0, 1], [0, 1, 1, 0], c=[0, 0, 1, 1], cmap="coolwarm", edgecolors="black", s=120, vmin=0, vmax=1)
left.set_title("your network's output over the input square")
right.plot(net["loss_history"], color="#8b5cf6")
right.set_title("loss per epoch"); right.set_xlabel("epoch")
plt.tight_layout(); plt.show()